# Seasonal Agriculture Performance Analysis

**VOIS AICTE Major Project — Data Visualization / Data Analytics**

### Project goal
Analyze agricultural performance across **Kharif, Rabi and Zaid** seasons and identify meaningful patterns in yield, production, revenue, cost, profit, water use, environmental conditions, crops, irrigation methods and regions.

### Workflow
1. Data loading and understanding
2. Data cleaning and preparation
3. Exploratory data analysis
4. Seasonal performance comparison
5. Profitability analysis
6. Irrigation analysis
7. Crop × season analysis
8. State-wise analysis
9. Correlation analysis
10. Statistical testing
11. Key findings and recommendations


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11


## 1. Load the dataset

In [ ]:
# Load dataset
df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


In [ ]:
# First five records
df.head()


In [ ]:
# Dataset structure
df.info()


In [ ]:
# Descriptive statistics
df.describe().T


## 2. Data quality and cleaning

In [ ]:
# Check missing values
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().sum() / len(df) * 100).round(2)
})

missing[missing["Missing_Count"] > 0].sort_values("Missing_Count", ascending=False)


In [ ]:
# Check duplicate records
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)


In [ ]:
# Remove exact duplicate rows, if any
clean_df = df.drop_duplicates().copy()

print("Original shape:", df.shape)
print("Cleaned shape:", clean_df.shape)


In [ ]:
# Check data types
clean_df.dtypes


In [ ]:
# Convert categorical columns to category type
categorical_columns = [
    "State", "District", "Crop", "Season", "Irrigation_Method"
]

for column in categorical_columns:
    clean_df[column] = clean_df[column].astype("category")

clean_df[categorical_columns].dtypes


In [ ]:
# Check category values
for column in categorical_columns:
    print("\n", "=" * 50)
    print(column)
    print("=" * 50)
    print(clean_df[column].value_counts())


In [ ]:
# Check negative values in numerical columns
numeric_columns = clean_df.select_dtypes(include=np.number).columns

negative_values = {
    column: int((clean_df[column] < 0).sum())
    for column in numeric_columns
    if (clean_df[column] < 0).sum() > 0
}

print("Negative-value counts:", negative_values)


### Missing-value treatment

The dataset contains missing values in **Rainfall_mm, Soil_Moisture_pct and Yield_Tonnes_Ha**.  
For descriptive analysis, pandas aggregation functions ignore missing observations automatically. We therefore retain valid observations instead of blindly deleting rows. For analyses that specifically require Yield, the code below uses non-missing Yield observations.

This keeps the analysis transparent and avoids unnecessary loss of data.


## 3. Basic exploratory analysis

In [ ]:
# Dataset dimensions and category counts
print("Number of farms:", clean_df["Farm_ID"].nunique())
print("States:", clean_df["State"].nunique())
print("Districts:", clean_df["District"].nunique())
print("Crops:", clean_df["Crop"].nunique())
print("Seasons:", clean_df["Season"].nunique())
print("Irrigation methods:", clean_df["Irrigation_Method"].nunique())


In [ ]:
# Season distribution
season_counts = clean_df["Season"].value_counts()

plt.figure()
season_counts.plot(kind="bar")
plt.title("Number of Farms by Season")
plt.xlabel("Season")
plt.ylabel("Number of farms")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Crop distribution
crop_counts = clean_df["Crop"].value_counts()

plt.figure(figsize=(10, 5))
crop_counts.plot(kind="bar")
plt.title("Number of Farms by Crop")
plt.xlabel("Crop")
plt.ylabel("Number of farms")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# State distribution
state_counts = clean_df["State"].value_counts()

plt.figure(figsize=(10, 5))
state_counts.plot(kind="bar")
plt.title("Number of Farms by State")
plt.xlabel("State")
plt.ylabel("Number of farms")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 4. Seasonal performance analysis

In [ ]:
# Put seasons in logical order
season_order = ["Kharif", "Rabi", "Zaid"]

clean_df["Season"] = pd.Categorical(
    clean_df["Season"],
    categories=season_order,
    ordered=True
)

season_summary = clean_df.groupby("Season", observed=True).agg(
    Farms=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Production=("Production_Tonnes", "mean"),
    Average_Revenue=("Revenue_INR", "mean"),
    Average_Cost=("Total_Cost_INR", "mean"),
    Average_Profit=("Profit_INR", "mean"),
    Average_Water=("Water_Used_m3", "mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Rainfall=("Rainfall_mm", "mean"),
    Temperature=("Avg_Temperature_C", "mean"),
    Disease_Pest_Risk=("Disease_Pest_Risk_pct", "mean")
)

season_summary


In [ ]:
# Profitability percentage by season
positive_profit = (
    clean_df.groupby("Season", observed=True)["Profit_INR"]
    .apply(lambda x: (x > 0).mean() * 100)
)

print("Percentage of farms with positive profit:")
print(positive_profit.round(2))


In [ ]:
# Seasonal yield comparison
plt.figure()
plt.bar(season_summary.index.astype(str), season_summary["Average_Yield"])
plt.title("Average Yield by Season")
plt.xlabel("Season")
plt.ylabel("Yield (tonnes/hectare)")
plt.tight_layout()
plt.show()


In [ ]:
# Seasonal profit comparison
plt.figure()
plt.bar(season_summary.index.astype(str), season_summary["Average_Profit"] / 1000)
plt.axhline(0, linewidth=1)
plt.title("Average Profit by Season")
plt.xlabel("Season")
plt.ylabel("Average Profit (₹ thousand)")
plt.tight_layout()
plt.show()


In [ ]:
# Revenue versus cost
x = np.arange(len(season_summary))
width = 0.35

plt.figure()
plt.bar(x - width/2, season_summary["Average_Revenue"] / 100000, width, label="Revenue")
plt.bar(x + width/2, season_summary["Average_Cost"] / 100000, width, label="Cost")
plt.xticks(x, season_summary.index.astype(str))
plt.ylabel("₹ lakh")
plt.title("Average Revenue and Cost by Season")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Irrigation method analysis

In [ ]:
irrigation_summary = clean_df.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Profit=("Profit_INR", "mean"),
    Average_Water=("Water_Used_m3", "mean"),
    Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Average_Profit", ascending=False)

irrigation_summary


In [ ]:
# Average profit by irrigation method
plt.figure()
plt.barh(
    irrigation_summary.index.astype(str),
    irrigation_summary["Average_Profit"] / 1000
)
plt.gca().invert_yaxis()
plt.xlabel("Average profit (₹ thousand)")
plt.title("Average Profit by Irrigation Method")
plt.tight_layout()
plt.show()


In [ ]:
# Water use and water efficiency by irrigation method
x = np.arange(len(irrigation_summary))
width = 0.35

fig, ax1 = plt.subplots()
ax1.bar(x - width/2, irrigation_summary["Average_Water"], width, label="Water used (m³)")
ax1.set_ylabel("Average water used (m³)")
ax1.set_xticks(x, irrigation_summary.index.astype(str), rotation=20)

ax2 = ax1.twinx()
ax2.bar(x + width/2, irrigation_summary["Water_Efficiency"], width, label="Water efficiency")
ax2.set_ylabel("Water efficiency (t/1000 m³)")

plt.title("Water Use and Efficiency by Irrigation Method")
plt.tight_layout()
plt.show()


## 6. Crop × season analysis

In [ ]:
crop_season_profit = clean_df.pivot_table(
    index="Crop",
    columns="Season",
    values="Profit_INR",
    aggfunc="mean"
).reindex(columns=season_order)

crop_season_profit


In [ ]:
# Crop-season profit heatmap using matplotlib
matrix = crop_season_profit.values / 1000

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(matrix, aspect="auto")

ax.set_xticks(range(len(crop_season_profit.columns)))
ax.set_xticklabels(crop_season_profit.columns.astype(str))
ax.set_yticks(range(len(crop_season_profit.index)))
ax.set_yticklabels(crop_season_profit.index)

ax.set_xlabel("Season")
ax.set_ylabel("Crop")
ax.set_title("Average Profit by Crop and Season (₹ thousand)")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if not np.isnan(matrix[i, j]):
            ax.text(j, i, f"{matrix[i,j]:.0f}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Profit (₹ thousand)")
plt.tight_layout()
plt.show()


In [ ]:
# Find best and worst crop-season combinations
crop_season_long = (
    clean_df.groupby(["Season", "Crop"], observed=True)["Profit_INR"]
    .mean()
    .reset_index()
)

best_combination = crop_season_long.loc[crop_season_long["Profit_INR"].idxmax()]
worst_combination = crop_season_long.loc[crop_season_long["Profit_INR"].idxmin()]

print("Best crop-season combination:")
print(best_combination)

print("\nWorst crop-season combination:")
print(worst_combination)


## 7. State-wise analysis

In [ ]:
state_summary = clean_df.groupby("State").agg(
    Farms=("Farm_ID", "count"),
    Average_Yield=("Yield_Tonnes_Ha", "mean"),
    Average_Revenue=("Revenue_INR", "mean"),
    Average_Profit=("Profit_INR", "mean")
).sort_values("Average_Profit", ascending=False)

state_summary


In [ ]:
# State-wise average profit
plt.figure(figsize=(10, 5))
plt.barh(
    state_summary.index,
    state_summary["Average_Profit"] / 1000
)
plt.gca().invert_yaxis()
plt.xlabel("Average profit (₹ thousand)")
plt.title("Average Profit by State")
plt.tight_layout()
plt.show()


## 8. Environmental and risk analysis

In [ ]:
# Seasonal environmental conditions
environment_summary = clean_df.groupby("Season", observed=True).agg(
    Rainfall_mm=("Rainfall_mm", "mean"),
    Temperature_C=("Avg_Temperature_C", "mean"),
    Humidity_pct=("Humidity_pct", "mean"),
    Soil_Moisture_pct=("Soil_Moisture_pct", "mean"),
    Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
)

environment_summary


In [ ]:
# Temperature by season
plt.figure()
plt.bar(environment_summary.index.astype(str), environment_summary["Temperature_C"])
plt.title("Average Temperature by Season")
plt.xlabel("Season")
plt.ylabel("Temperature (°C)")
plt.tight_layout()
plt.show()


In [ ]:
# Rainfall by season
plt.figure()
plt.bar(environment_summary.index.astype(str), environment_summary["Rainfall_mm"])
plt.title("Average Rainfall by Season")
plt.xlabel("Season")
plt.ylabel("Rainfall (mm)")
plt.tight_layout()
plt.show()


## 9. Correlation analysis

In [ ]:
# Correlation of numerical variables with profit
correlation_with_profit = (
    clean_df.select_dtypes(include=np.number)
    .corr()["Profit_INR"]
    .sort_values(ascending=False)
)

correlation_with_profit


In [ ]:
# Plot strongest absolute correlations with profit
corr_without_profit = correlation_with_profit.drop("Profit_INR")
top_corr = corr_without_profit.abs().sort_values(ascending=False).head(8)
plot_values = correlation_with_profit[top_corr.index].sort_values()

plt.figure(figsize=(9, 5))
plt.barh(plot_values.index, plot_values.values)
plt.xlabel("Correlation with Profit")
plt.title("Variables Most Associated with Profit")
plt.tight_layout()
plt.show()


## 10. Statistical testing

### One-way ANOVA

We test whether the mean outcome differs across **Kharif, Rabi and Zaid**.

- **H₀:** Mean outcome is the same across all seasons.
- **H₁:** At least one seasonal mean is different.
- Significance level: **α = 0.05**.

A significant p-value means there is evidence of a difference in means, but it does **not** by itself prove that season causes the difference.


In [ ]:
# ANOVA: Profit across seasons
profit_groups = [
    group["Profit_INR"].dropna().values
    for _, group in clean_df.groupby("Season", observed=True)
]

profit_anova = stats.f_oneway(*profit_groups)

print("ANOVA for Profit")
print("F-statistic:", round(profit_anova.statistic, 4))
print("p-value:", profit_anova.pvalue)

if profit_anova.pvalue < 0.05:
    print("Result: Reject H0 — mean profit differs significantly across seasons.")
else:
    print("Result: Fail to reject H0.")


In [ ]:
# ANOVA: Yield across seasons
yield_groups = [
    group["Yield_Tonnes_Ha"].dropna().values
    for _, group in clean_df.groupby("Season", observed=True)
]

yield_anova = stats.f_oneway(*yield_groups)

print("ANOVA for Yield")
print("F-statistic:", round(yield_anova.statistic, 4))
print("p-value:", yield_anova.pvalue)

if yield_anova.pvalue < 0.05:
    print("Result: Reject H0 — mean yield differs significantly across seasons.")
else:
    print("Result: Fail to reject H0 — evidence is insufficient to conclude a seasonal mean-yield difference.")


## 11. Automated key findings

In [ ]:
# Generate findings directly from the analysis

best_profit_season = season_summary["Average_Profit"].idxmax()
worst_profit_season = season_summary["Average_Profit"].idxmin()

best_yield_season = season_summary["Average_Yield"].idxmax()
worst_yield_season = season_summary["Average_Yield"].idxmin()

best_irrigation = irrigation_summary["Average_Profit"].idxmax()
best_efficiency_irrigation = irrigation_summary["Water_Efficiency"].idxmax()
highest_water_irrigation = irrigation_summary["Average_Water"].idxmax()

print("KEY FINDINGS")
print("-" * 70)
print(f"1. Highest average profit: {best_profit_season} "
      f"(₹{season_summary.loc[best_profit_season, 'Average_Profit']:,.0f}).")

print(f"2. Lowest average profit: {worst_profit_season} "
      f"(₹{season_summary.loc[worst_profit_season, 'Average_Profit']:,.0f}).")

print(f"3. Highest average yield: {best_yield_season} "
      f"({season_summary.loc[best_yield_season, 'Average_Yield']:.2f} t/ha).")

print(f"4. Lowest average yield: {worst_yield_season} "
      f"({season_summary.loc[worst_yield_season, 'Average_Yield']:.2f} t/ha).")

print(f"5. Most profitable irrigation method: {best_irrigation} "
      f"(₹{irrigation_summary.loc[best_irrigation, 'Average_Profit']:,.0f} average profit).")

print(f"6. Highest water-efficiency method: {best_efficiency_irrigation} "
      f"({irrigation_summary.loc[best_efficiency_irrigation, 'Water_Efficiency']:.2f} t/1000 m³).")

print(f"7. Highest average water use: {highest_water_irrigation} "
      f"({irrigation_summary.loc[highest_water_irrigation, 'Average_Water']:,.0f} m³).")

print(f"8. Profit ANOVA p-value: {profit_anova.pvalue:.3e}.")
print(f"9. Yield ANOVA p-value: {yield_anova.pvalue:.3f}.")


## 12. Conclusions and recommendations

### Conclusions
- Compare seasons using both **productivity and economics**; yield alone is not enough.
- Crop choice creates substantial differences inside the same season.
- Irrigation methods show different trade-offs between yield, profit and water efficiency.
- State-level averages indicate that location also matters.
- Correlation and ANOVA identify useful relationships/differences, but they do not prove causation.

### Recommendations
1. Prioritize crop–season combinations with consistently positive profitability.
2. Investigate the economics of Zaid before expanding production in that season.
3. Evaluate drip irrigation where suitable, considering both investment cost and observed productivity.
4. Reduce unnecessary flood irrigation where agronomically feasible because it has high observed water use and lower water efficiency.
5. Build district-level dashboards for more localized planning.
6. For future work, use regression/multivariate analysis to control for crop, region, farm size, irrigation, climate and input variables.


## 13. Final project checklist

- [x] Dataset loaded
- [x] Dataset structure explored
- [x] Missing values checked
- [x] Duplicate records checked
- [x] Categorical variables prepared
- [x] Seasonal analysis completed
- [x] Yield / production analysis completed
- [x] Revenue / cost / profit analysis completed
- [x] Irrigation analysis completed
- [x] Crop × season analysis completed
- [x] State analysis completed
- [x] Environmental analysis completed
- [x] Correlation analysis completed
- [x] Statistical testing completed
- [x] Findings and recommendations generated
